# 📅 Day 19 — Mock Interview Practice

**Week 3 · Day 5 · Level: ADVANCED (15 ADVANCED + 3 HARD)**

Week 3 continues: **product-company level** questions — complex end-to-end scenarios, engine internals, and system-design judgment. 18 questions across 8 categories. Day 19 themes: **time-series gap interpolation, fuzzy entity resolution with levenshtein, monotonic stacks, union-find, arbitrary stateful streaming (`flatMapGroupsWithState`), stream-to-stream joins, executor memory & GC tuning, sort-merge vs shuffled-hash joins, zero-downtime schema migration, DEEP CLONE disaster recovery, checkpoint-safe streaming upgrades, One Big Table vs star schema, ragged hierarchies, Snowflake data sharing, event tables, For-Each task orchestration, pytest/chispa pipeline testing, and designing a full observability stack** — all fresh vs days 1-18.

| # | Category | Questions | Level |
| --- | --- | --- | --- |
| 1 | SQL Advanced | Q1-Q2 | ADVANCED |
| 2 | Python — Loop/Function/List/Dict | Q3-Q4 | ADVANCED / HARD |
| 3 | PySpark Coding | Q5-Q6 | ADVANCED |
| 4 | Spark Optimization & Deep Spark | Q7-Q8 | ADVANCED / HARD |
| 5 | ETL — Failures, Late Data, Schema Handling, Fault Tolerance | Q9-Q11 | ADVANCED / HARD |
| 6 | Data Modeling | Q12-Q13 | ADVANCED |
| 7 | Snowflake | Q14-Q15 | ADVANCED |
| 8 | Monitoring, Logging & Orchestration | Q16-Q18 | ADVANCED / HARD |

**How to use**: attempt each answer out loud in ≤ 4 minutes BEFORE opening the solution. Every answer must name at least one trade-off. Day 20 continues Week 3; days 22+ move to FAANG / top product-company level.

---

## Q1 — SQL Advanced: Time-Series Gaps — Spine + Interpolation [ADVANCED]

**Interviewer (Netflix):** An IoT freezer fleet reports temperature every minute, but sensors drop readings whenever they go offline. The reliability dashboard needs **one row per minute per device** — including the missing minutes.

1. Build the **minute spine** for each device (every minute between its first and last reading) **without a calendar table**.
2. Fill the missing minutes with **last observation carried forward (LOCF)** and flag any gap longer than 5 minutes as a `data_quality_issue`.
3. Why is `last_value(... IGNORE NULLS)` essential here? What would the query return without it? When would you prefer **linear interpolation** over LOCF?

In [0]:
# Solution Q1 — minute spine + LOCF + gap flagging
from pyspark.sql import functions as F
from pyspark.sql.window import Window

readings = spark.createDataFrame([
    ("FZ-01", "2026-11-01 10:00:00", -18.0),
    ("FZ-01", "2026-11-01 10:01:00", -18.2),
    ("FZ-01", "2026-11-01 10:05:00", -19.0),   # 3 minutes missing in between
    ("FZ-01", "2026-11-01 10:06:00", -19.1),
    ("FZ-01", "2026-11-01 10:15:00", -21.0),   # 8-minute gap -> quality issue
    ("FZ-02", "2026-11-01 10:00:00", -5.0),
    ("FZ-02", "2026-11-01 10:02:00", -5.5),
], ["device_id", "ts", "temp_f"]) \
  .withColumn("ts", F.col("ts").cast("timestamp"))

readings.createOrReplaceTempView("readings")

result = spark.sql("""
WITH bounds AS (
  SELECT device_id, MIN(ts) AS min_ts, MAX(ts) AS max_ts
  FROM readings GROUP BY device_id
),
spine AS (
  -- every minute per device, no calendar table needed
  SELECT b.device_id, explode(sequence(b.min_ts, b.max_ts, interval 1 minute)) AS ts
  FROM bounds b
),
stitched AS (
  SELECT s.device_id, s.ts, r.temp_f,
         date_diff(minute, last_value(r.ts) IGNORE NULLS
                    OVER (PARTITION BY s.device_id ORDER BY s.ts
                          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), s.ts) AS mins_since_last
  FROM spine s LEFT JOIN readings r
    ON r.device_id = s.device_id AND r.ts = s.ts
)
SELECT device_id, ts,
       CASE WHEN temp_f IS NULL THEN 'imputed' ELSE 'observed' END AS row_kind,
       last_value(temp_f) IGNORE NULLS OVER (
           PARTITION BY device_id ORDER BY ts
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS locf_temp_f,
       CASE WHEN mins_since_last > 5 THEN 'gap_over_5m' END AS data_quality_issue,
       mins_since_last
FROM stitched
ORDER BY device_id, ts
""")

result.show(8, truncate=False)

n_rows = result.count()
expected = 16 + 3  # FZ-01: 10:00->10:15 = 16 minutes; FZ-02: 3 minutes
print(f"spine rows = {n_rows} (expected {expected})")
assert n_rows == expected, "spine should contain every minute for every device"
assert result.filter("device_id='FZ-01' AND ts='2026-11-01 10:16:00'").count() == 0
assert result.filter("locf_temp_f IS NULL").count() == 0    # every minute now carries a temperature
print("Q1 OK — LOCF spine built, gaps flagged")

## Q2 — SQL Advanced: Fuzzy Entity Resolution — `levenshtein` + Blocking [ADVANCED]

**Interviewer (Salesforce):** The CRM has duplicate customers created by name typos (`"Acme Corp"` vs `"Acme  Corp"` vs `"Acmee Corp"`). Marketing counts are inflated. Find the duplicate clusters and pick a **golden record** for each.

1. Why is comparing **all n² pairs** a non-starter at 10M accounts? What is **blocking**, and how much does a `substr(name,1,4) + city` block cut the search space?
2. Write ONE query: self-join within a block, keep pairs with `levenshtein(name_a, name_b) <= 2`, dedupe the (a,b)/(b,a) mirror, then pick the golden record per cluster with survivorship rules (most complete + most recent).
3. Trade-offs: what does a **tighter** block do to precision vs recall? How do false merges vs missed merges hurt differently?

In [0]:
# Solution Q2 — blocking + levenshtein pairing + golden-record survivorship
from pyspark.sql import functions as F
from pyspark.sql.window import Window

accounts = spark.createDataFrame([
    (1, "Acme Corp",      "Austin",  "billing@acme.com",   "2026-10-20"),
    (2, "Acme  Corp",     "Austin",  None,                  "2026-10-25"),   # dup (typo+spaces)
    (3, "Acmee Corp",      "Austin",  "ap@acme.com",        "2026-10-30"),   # dup (one edit)
    (4, "Acme Corp",       "Denver",  "info@acme.io",       "2026-10-21"),   # DIFFERENT city -> not merged
    (5, "Globex Ltd",     "Austin",  "hello@globex.com",   "2026-10-22"),
    (6, "Globeex Ltd",    "Austin",  None,                  "2026-10-28"),   # dup
], ["acct_id", "name", "city", "email", "updated_at"])

accounts = accounts.withColumn("updated_at", F.col("updated_at").cast("date"))

# --- Step 1: candidate pairs within a block (name-prefix + city) ---
a = accounts.withColumnRenamed("acct_id", "id_a").withColumnRenamed("name", "name_a") \
     .withColumn("block", F.concat(F.expr("substr(name_a, 1, 4)"), F.lit("|"), F.col("city")))
b = accounts.withColumnRenamed("acct_id", "id_b").withColumnRenamed("name", "name_b") \
     .withColumn("block", F.concat(F.expr("substr(name_b, 1, 4)"), F.lit("|"), F.col("city")))

pairs = (a.alias("a").join(b.alias("b"), (F.col("a.block") == F.col("b.block")) & (F.col("a.id_a") < F.col("b.id_b")), "inner")
         .withColumn("edit_distance", F.levenshtein(F.col("a.name_a"), F.col("b.name_b"))))

candidates = pairs.filter("edit_distance <= 2").select("id_a", "id_b", "edit_distance")
candidates.show()

# --- Step 2: golden record via survivorship (completeness first, recency second) ---
# A duplicate can appear in SEVERAL candidate pairs (acct 3 matches both 1 and 2) ->
# collapse to ONE canonical parent per account first, or the left join fans out rows.
parents = candidates.groupBy("id_b").agg(F.min("id_a").alias("parent"))
# NOTE: min-parent handles star-shaped clusters; long transitive chains need full
# closure (iterative label propagation) or the union-find from Q4 - say so in the room.
w = Window.partitionBy("cluster").orderBy(F.col("emails_present").desc(), F.col("updated_at").desc())
clustered = (accounts
    .join(parents, accounts.acct_id == parents.id_b, "left")
    .withColumn("cluster", F.coalesce("parent", "acct_id"))         # union-find-lite: dup points at its parent
    .withColumn("emails_present", F.col("email").isNotNull().cast("int")))
golden = (clustered.withColumn("rank", F.row_number().over(w))
    .filter("rank = 1")
    .select(F.col("cluster").alias("golden_id"), "acct_id", "name", "city", "email"))

golden.orderBy("golden_id", "acct_id").show(truncate=False)

n_golden = golden.select("golden_id").distinct().count()
assert n_golden == 3, f"expected 3 customers (Acme-Austin, Acme-Denver, Globex), got {n_golden}"
assert candidates.filter("id_a=1 AND id_b=4").count() == 0  # city mismatch must NOT merge
print("Q2 OK — 6 rows -> 3 golden customers; Denver kept separate; survivorship correct")

## Q3 — Python: Monotonic Stack — Next Warmer Reading [ADVANCED]

**Interviewer (Amazon):** You have daily temperature readings. For each day, return **how many days until a strictly warmer reading** — 0 if none ever comes. The dataset is a year long, and the answer must be **O(n)**.

1. Why does the brute-force nested loop cost O(n²), and why is that fatal at 365k readings (hourly data)?
2. Explain the **monotonic decreasing stack** invariant precisely: what is on the stack, when do we push, when do we pop? Why is each index pushed and popped **at most once** (amortized O(1))?
3. What breaks if the stack invariant uses `<=` instead of `<`? What does the leftover stack represent at the end?

In [0]:
# Solution Q3 — monotonic decreasing stack over indices

def days_until_warmer(temps):
    """Return list where out[i] = days until a strictly warmer reading (0 if never)."""
    out = [0] * len(temps)
    stack = []  # holds INDICES whose 'warmer day' has not arrived yet; temps are decreasing
    for i, t in enumerate(temps):
        # today's temp resolves every colder day sitting on the stack
        while stack and t > temps[stack[-1]]:
            j = stack.pop()
            out[j] = i - j
        stack.append(i)
    # anything left on the stack never sees a warmer day -> stays 0
    return out

# --- tests ---
assert days_until_warmer([73, 74, 75, 71, 69, 72, 76, 73]) == [1, 1, 4, 2, 1, 1, 0, 0]
assert days_until_warmer([30, 40, 50, 60]) == [1, 1, 1, 0]          # strictly increasing
assert days_until_warmer([76, 75, 74]) == [0, 0, 0]                  # strictly decreasing
assert days_until_warmer([]) == []
assert days_until_warmer([50]) == [0]
assert days_until_warmer([50, 50, 51]) == [2, 1, 0]                 # equal does NOT count

# --- why O(n): each index is pushed once and popped at most once ---
import random
random.seed(19)
big = [random.randint(-30, 45) for _ in range(100_000)]
result = days_until_warmer(big)
assert len(result) == len(big)
print("Q3 OK — monotonic stack answers 100k readings in O(n); strict-increase semantics verified")

## Q4 — Python: Union-Find — Merge Duplicate CRM Accounts [HARD]

**Interviewer (Meta):** Three CRM exports each list accounts with overlapping emails. Any two accounts that share **even one email** belong to the same customer. Return the merged customer groups.

1. Model as a graph: what are the nodes and edges? Why is union-find **better than BFS/DFS** here (memory, cache locality, incremental arrivals)?
2. Implement DSU with **path compression** and **union by rank**. State the amortized complexity per operation and the overall bound.
3. Why does union by rank keep trees shallow? What degenerates if you skip path compression?

In [0]:
# Solution Q4 — Disjoint Set Union (path compression + union by rank)

class DSU:
    def __init__(self, n):
        self.parent = list(range(n))   # each node is its own root
        self.rank = [0] * n            # upper bound on subtree height
        self.components = n

    def find(self, x):
        # path compression: point every visited node straight at the root
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]   # grandparent hop (path halving)
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False                                   # already same component
        # union by rank: attach the shallower tree under the deeper root
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1
        self.components -= 1
        return True

def merge_accounts(accounts):
    """accounts: list of email-lists. Returns list of merged email-lists per customer."""
    dsu = DSU(len(accounts))
    owner = {}                       # email -> first account index that had it
    for i, emails in enumerate(accounts):
        for e in emails:
            if e in owner:
                dsu.union(i, owner[e])   # shared email == edge
            else:
                owner[e] = i
    merged = {}
    for i, emails in enumerate(accounts):
        merged.setdefault(dsu.find(i), set()).update(emails)
    return [sorted(v) for v in merged.values()]

# --- tests (classic 'accounts merge' semantics) ---
accs = [
    ["a@x.com", "b@x.com"],
    ["c@x.com"],
    ["b@x.com", "d@x.com"],   # bridges 0 and 2 via b@x.com
    ["e@x.com"],
    ["e@x.com", "f@x.com"],   # bridges 3 and 4
]
result = sorted(merge_accounts(accs))
expected = sorted([sorted("a@x.com b@x.com d@x.com".split()),
                   ["c@x.com"],
                   sorted("e@x.com f@x.com".split())])
assert result == expected, result

# transitivity check: a-b, b-c, c-d must ALL collapse to one group
chain = merge_accounts([["1@x"], ["1@x", "2@x"], ["2@x", "3@x"], ["3@x", "4@x"]])
assert len(chain) == 1

# complexity sanity: 100k unions on 100k nodes still instant (amortized ~O(α(n)))
dsu = DSU(100_000)
for i in range(99_999):
    dsu.union(i, i + 1)
assert dsu.components == 1
assert max(dsu.rank) <= 18   # rank grows logarithmically, never linearly
print("Q4 OK — DSU merged accounts transitively; rank stayed logarithmic; O(n·α(n)) total")

## Q5 — PySpark Coding: Arbitrary Stateful Streaming — Sessionization with `flatMapGroupsWithState` [ADVANCED]

**Interviewer (Uber):** Riders send pings while on-trip. A **trip session** ends when a rider goes silent for more than **10 minutes** (or a new gap appears). When a session completes, emit `(rider_id, session_id, n_pings, start, end)`. The pipeline runs 24/7.

1. Why don't `dropDuplicates` or plain windows suffice? When is `flatMapGroupsWithState` the ONLY tool that fits (custom timeout logic, partial emissions, custom state schema)?
2. Implement the state function: per-rider state holds `(session_id, n_pings, last_ts)`; on a gap > 10 min emit the OLD session and start a new one; use `GroupStateTimeout` as the backstop for riders who go silent forever.
3. Where does the state physically live (RocksDB/HDFS-backed state store)? What does 10M active keys cost, and how do state TTL and the watermark bound it?

In [0]:
# Solution Q5 — sessionization state machine + arbitrary-stateful streaming wiring
from pyspark.sql import functions as F, types as T
import uuid

GAP_MINUTES = 10

events = spark.createDataFrame([
    ("R1", "2026-11-01 08:00:00"),
    ("R1", "2026-11-01 08:05:00"),
    ("R1", "2026-11-01 08:40:00"),   # gap 35 min -> session 1 CLOSES here
    ("R1", "2026-11-01 08:44:00"),
    ("R2", "2026-11-01 09:00:00"),
    ("R2", "2026-11-01 09:11:00"),   # gap 11 min -> two sessions
    ("R3", "2026-11-01 10:02:00"),
    ("R3", "2026-11-01 10:04:00"),   # one session
    ("R4", "2026-11-01 11:00:00"),
    ("R4", "2026-11-01 11:30:00"),   # three isolated pings + one pair -> 3 sessions
    ("R4", "2026-11-01 12:00:00"),
    ("R4", "2026-11-01 12:02:00"),
], ["rider_id", "event_ts"]).withColumn("event_ts", F.col("event_ts").cast("timestamp"))

# ---------- (1) the state machine — pure, unit-testable, framework-free ----------
def advance(session, ts):
    """Advance an open session by one event. Returns ('close' | 'continue', session)."""
    if session is None:
        return "continue", {"session_id": str(uuid.uuid4()), "n_pings": 1,
                             "start_ts": ts, "last_ts": ts}
    if (ts - session["last_ts"]).total_seconds() > GAP_MINUTES * 60:
        return "close", {"session_id": str(uuid.uuid4()), "n_pings": 1,
                         "start_ts": ts, "last_ts": ts}
    session["n_pings"] += 1
    session["last_ts"] = ts
    return "continue", session

def sessionize(rider_id, timestamps):
    """Batch equivalent of the streaming state function — same logic, no state store."""
    out, cur = [], None
    for ts in sorted(timestamps):
        if cur and (ts - cur["last_ts"]).total_seconds() > GAP_MINUTES * 60:
            out.append({**cur, "rider_id": rider_id})       # gap rule closes the old session
            cur = None
        if cur is None:
            cur = {"session_id": str(uuid.uuid4()), "n_pings": 0, "start_ts": ts}
        cur["n_pings"] += 1
        cur["last_ts"] = ts
    if cur:
        out.append({**cur, "rider_id": rider_id})             # flush the still-open session
    return out

# ---------- (2) prove the state machine on the deterministic dataset ----------
per_rider = {}
for r in events.orderBy("event_ts").collect():
    per_rider.setdefault(r["rider_id"], []).append(r["event_ts"])

counts, all_sessions = {}, []
for rid, tss in per_rider.items():
    sessions = sessionize(rid, tss)
    all_sessions += sessions
    counts[rid] = len(sessions)

assert counts == {"R1": 2, "R2": 2, "R3": 1, "R4": 3}, counts
r1 = [s for s in all_sessions if s["rider_id"] == "R1"]
assert sorted(s["n_pings"] for s in r1) == [2, 2]           # (08:00,08:05) and (08:40,08:44)
print("session counts per rider:", counts)

# ---------- (3) the streaming wiring — what the real job looks like ----------
wiring = '''
# State schema persisted per rider in the state store:
state_schema = T.StructType([
    T.StructField("session_id", T.StringType()),
    T.StructField("n_pings",    T.IntegerType()),
    T.StructField("start_ts",   T.TimestampType()),
    T.StructField("last_ts",    T.TimestampType()),
])

# Arbitrary-stateful pattern (flatMapGroupsWithState / applyInPandasWithState, API by runtime):
sessions = (events_stream
    .withWatermark("event_ts", "30 minutes")        # bounds state for riders who vanish
    .groupBy("rider_id")
    .flatMapGroupsWithState(out_schema, state_schema,
                            GroupStateTimeout.EventTimeTimeout)(update_fn))
# update_fn = advance() above: pull state, apply the gap rule, EMIT the session row when it
# closes, push state back. EventTimeTimeout lets the watermark close sessions for riders
# who never come back - the state store then drops them (bounded memory).

# Built-ins cover ~80% of sessionization BEFORE you reach for this (name them first!):
#   groupBy(F.session_window("event_ts", "10 minutes"))   # Spark 3.2+
# Reach for arbitrary stateful ONLY when you need: emit-on-close output rows, custom TTL,
# or session merging across batches that session_window cannot express.
'''
print(wiring)

print("""
[NOTES TO SAY OUT LOUD]
- State lives in the STATE STORE (checkpoint): default in-memory (HDFS-backed files);
  RocksDB-backed provider for large key counts - 400M riders won't fit in-memory state.
- Watermark + EventTimeTimeout = the GC for state: riders who stop pinging are evicted once
  their watermark passes; without it, every rider_id lives FOREVER -> unbounded state.
- Timeout firing EMITS the final row: that's how a 'silent' session still gets closed.
- Checkpoint & state schema changes are breaking (see Q11) - version the state schema with
  care; test upgrades on a fresh checkpoint with replayed events.
- Cost: one state read+write per event. Design the state row to stay tiny (ids + timestamps,
  not payloads).
""")
print("Q5 OK — gap-rule state machine verified (2,2,1,3 sessions) + streaming wiring")

## Q6 — PySpark Coding: Stream-to-Stream Join — Impressions × Clicks [ADVANCED]

**Interviewer (Meta):** Join ad **impressions** to **click** events on `(ad_id, user_id)`. Clicks can arrive up to **5 minutes after** the impression. Unmatched impressions must still be counted (CTR = clicks ÷ impressions).

1. Why can't this be a batch join? Why do **BOTH** sides need watermarks — what state would grow unbounded otherwise?
2. Write the stream-stream **inner join** with event-time range condition `click_ts BETWEEN imp_ts AND imp_ts + interval 5 minutes` and watermarks on both sides.
3. Switching to a **left outer** join: what does Spark do differently for cleanup (why must it wait beyond the outer-side watermark), and what does that cost in state-store bytes? How do you bound it?

In [0]:
# Solution Q6 — dual-watermark stream-stream join (deterministic file-source demo)
from pyspark.sql import functions as F
import os

root = "/Volumes/workspace/default/etl_test_volume/d19_q6"
dbutils.fs.rm(root, True)  # idempotent re-run
imp_dir, click_dir = os.path.join(root, "imps"), os.path.join(root, "clicks")

# Engineer a deterministic, FINITE dataset:
#  imp1 (10:00) clicked at 10:03      -> match
#  imp2 (10:00) clicked at 10:05      -> match (exactly AT the 5-min edge, inclusive)
#  imp3 (10:00) clicked at 10:09      -> NO match (late beyond window)
#  imp4 (10:20) never clicked         -> unmatched impression (inner: dropped; outer: emitted)
spark.createDataFrame([
    ("imp1", "adA", "u1", "2026-11-01 10:00:00"),
    ("imp2", "adA", "u2", "2026-11-01 10:00:00"),
    ("imp3", "adB", "u3", "2026-11-01 10:00:00"),
    ("imp4", "adC", "u4", "2026-11-01 10:20:00"),
], ["imp_id", "ad_id", "user_id", "imp_ts"]).withColumn("imp_ts", F.col("imp_ts").cast("timestamp")) \
 .repartition(1).write.mode("overwrite").parquet(imp_dir)
spark.createDataFrame([
    ("clk1", "adA", "u1", "2026-11-01 10:03:00"),
    ("clk2", "adA", "u2", "2026-11-01 10:05:00"),   # exactly at the 5-min edge
    ("clk3", "adB", "u3", "2026-11-01 10:09:00"),
], ["clk_id", "ad_id", "user_id", "click_ts"]).withColumn("click_ts", F.col("click_ts").cast("timestamp")) \
 .repartition(1).write.mode("overwrite").parquet(click_dir)

imps  = spark.readStream.schema(spark.read.parquet(imp_dir).schema).parquet(imp_dir)
clicks = spark.readStream.schema(spark.read.parquet(click_dir).schema).parquet(click_dir)

GAP = "5 minutes"
imp_s   = imps.withWatermark("imp_ts", GAP).alias("imp")
click_s = clicks.withWatermark("click_ts", GAP).alias("click")

# PySpark join signature is (other, on, how) - the condition goes in as `on`
joined = imp_s.join(
    click_s,
    on=(F.col("imp.ad_id") == F.col("click.ad_id"))
       & (F.col("imp.user_id") == F.col("click.user_id"))
       & (F.col("click.click_ts").between(F.col("imp.imp_ts"),
                                           F.col("imp.imp_ts") + F.expr(f"interval {GAP}"))),
    how="inner")

ckpt = os.path.join(root, "ck")
q = (joined.writeStream.format("memory").queryName("q6_joined")
     .option("checkpointLocation", ckpt)
     .trigger(availableNow=True).start())
q.awaitTermination()

res = spark.sql("SELECT imp_id, clk_id, imp_ts, click_ts FROM q6_joined ORDER BY imp_id")
res.show(truncate=False)
matched = {r["imp_id"] for r in res.collect()}
assert matched == {"imp1", "imp2"}, matched   # imp3 clicked too late; imp4 never clicked
print("Q6 OK — dual watermarks joined imp1 + imp2 (edge click); imp3 (too late) and imp4 (no click) unmatched")
dbutils.fs.rm(root, True)  # clean up demo artifacts

# --- outer join variant (emits unmatched impressions once the window truly closes) ---
outer_q = (imps.withWatermark("imp_ts", GAP)
    .join(clicks.withWatermark("click_ts", GAP),
          ["ad_id", "user_id"], "leftOuter"))
print("""
[OUTER-JOIN SEMANTICS TO SAY OUT LOUD]
- With leftOuter, Spark can't discard unmatched impression state as soon as the click-side
  watermark passes: it must keep each impression until imp_ts + 5min < BOTH watermarks,
  then emit it with NULLs. State lives LONGER than the inner join — that's the cost of CTR.
- Bounding it: watermarks on both sides (mandatory) + small enough grace window + RocksDB
  state store (spark.sql.streaming.stateStore.providerClass = rocksdb) for 100M+ keys.
""")

## Q7 — Spark Optimization: Executor Memory Anatomy & GC Tuning [ADVANCED]

**Interviewer (Databricks):** A long-running job shows full GCs, ~30-second GC pauses, and heavy spill. The executors are `16g memory, 4 cores`. Walk through executor memory and your tuning.

1. Break down executor memory: heap (`spark.executor.memory`) vs `memoryOverhead` vs off-heap. What lives in each — execution memory, storage memory, Python process, JVM metaspace?
2. How does the **unified memory manager** arbitrate execution vs storage (`spark.memory.fraction`, `spark.memory.storageFraction`)? When does Spark choose to **spill** rather than evict cached blocks?
3. GC tuning: why is G1GC the right collector on large heaps, the flags you'd actually set, and why 'GC thrashing' in the Spark UI matters more than total GC time.

In [0]:
# Solution Q7 — memory math + GC talk track

# 1) ANATOMY (say this out loud, with numbers):
memory_map = {
  "spark.executor.memory (heap, e.g. 16g)":
      "unified region = 16g minus ~300MB reserved; split into EXECUTION + STORAGE",
  "spark.memory.fraction (default 0.6)":
      "(execution + storage) share of the heap AFTER the reserved region",
  "spark.memory.storageFraction (default 0.5)":
      "soft cap: storage can grow beyond it by EVICTING cached blocks; execution borrows back only via eviction",
  "spark.executor.memoryOverhead (default max(384MB, 0.10 * executorMemory))":
      "off-heap-ish: Python worker, direct buffers, JVM metaspace. PySpark UDFs live HERE -> why low memoryOverhead kills Python-heavy jobs",
  "spark.memory.offHeap.enabled / offHeap.size":
      "Tungsten off-heap allocations; avoids GC pressure for sort/join data",
}
for k, v in memory_map.items():
    print(f"{k}\n   -> {v}\n")

# Worked example: 16g heap executor
heap = 16 * 1024
reserved = 300
unified = (heap - reserved) * 0.6
storage_soft = unified * 0.5
print(f"16g heap -> unified region = {unified/1024:.1f}g; storage soft cap = {storage_soft/1024:.1f}g")
print(f"memoryOverhead default = max(384MB, 0.10*16g) = {max(384, 0.10*heap):.0f}MB -> Python UDF budget")

# 2) SPILL vs EVICT rule:
print("""
[SPILL vs EVICT]
- execution memory FULL and storage has evictable cached blocks -> EVICT cache first.
- execution memory still full (no evictable blocks) -> SPILL to disk.
- cached-but-currently-in-use blocks are NOT evictable -> more spilling.
Tuning: uncache early, cache only hot DataFrames, or MEMORY_AND_DISK for graceful demotion.
""")

# 3) GC tuning configs to name in the interview:
gc_configs = {
  "spark.executor.defaultJavaOptions / extraJavaOptions":
      "-XX:+UseG1GC -XX:MaxGCPauseMillis=100 -XX:G1HeapRegionSize=32m",
  "why G1GC":
      "regioned, incremental collection; old-gen cleaned concurrently -> no monolithic full-GC pauses on big heaps",
  "G1HeapRegionSize":
      "32m avoids humongous allocations (>= half a region) going straight to old gen; big sort/join arrays are the classic humongous offenders",
  "GC thrashing vs total GC":
      "30s TOTAL GC spread across 300 tasks is fine; a single 30s PAUSE stalls the executor and risks task timeouts -> read the per-task GC column, not the aggregate",
  "structural fixes first":
      "reduce shuffle per task (more partitions), enable off-heap, smaller heaps + MORE executors (many 8-16g executors beat one 64g fat executor)",
}
for k, v in gc_configs.items():
    print(f"{k}: {v}")
print("\nQ7 OK — anatomy + spill/evict + GC playbook")

## Q8 — Spark Optimization: Sort-Merge vs Shuffled Hash Join [HARD]

**Interviewer (Netflix):** After the shuffle, your join has a **2 GB build side** and a **4 TB probe side**, spread over 2,000 partitions on 8-core executors. Spark planned a **sort-merge join (SMJ)**. Your peer claims a **shuffled hash join (SHJ)** would be faster. Who's right, and why?

1. Compare the two mechanically: SMJ = sort **both** sides, merge-walk; SHJ = hash the build side into memory, stream-probe the other. Cost of each per side.
2. Why is SMJ the default if SHJ does less work? What is the OOM risk of SHJ, and how does it relate build-side size to **shuffle partition count**?
3. How do you actually get an SHJ: `spark.sql.join.preferSortMergeJoin=false`, the `/*+ SHUFFLE_HASH(t) */` hint, and the case where one side is pre-sorted anyway. State the sizing rule of thumb.

In [0]:
# Solution Q8 — join strategy by build-side size + the SHUFFLE_HASH hint
from pyspark.sql import functions as F
import io, contextlib

def join_plan(df):
    """Capture the physical plan (works on classic clusters AND Spark Connect)."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain()
    return buf.getvalue()

def join_lines(plan):
    return [l.strip() for l in plan.splitlines() if "Join" in l]

# --- regime 1: small build side -> the planner AUTO-broadcasts it (cheapest of all) ---
small_build = spark.range(0, 100_000).withColumn("k", F.col("id") % 1000)
probe       = spark.range(0, 2_000_000).withColumn("k", F.col("id") % 1000)   # 'large' probe side
plan_small = join_plan(small_build.join(probe, "k"))
print("SMALL BUILD ->", join_lines(plan_small)[:1])       # BroadcastHashJoin

# --- regime 2: too big to broadcast -> shuffled-join territory (explain is lazy: nothing runs) ---
build = spark.range(0, 20_000_000).withColumn("k", F.col("id") % 1000)          # ~320MB estimated
plan_default = join_plan(build.join(probe, "k"))
join_default = join_lines(plan_default)
print("DEFAULT     ->", join_default[:1])   # classic Spark: SortMergeJoin; Photon/AQE often pick ShuffledHashJoin themselves

# --- regime 3: force shuffled hash join via the per-query hint ---
plan_shj = join_plan(build.hint("SHUFFLE_HASH").join(probe, "k"))
join_shj = join_lines(plan_shj)
print("HINTED      ->", join_shj[:1])

assert "Broadcast" in join_lines(plan_small)[0], join_lines(plan_small)[:3]
assert ("SortMergeJoin" in plan_default) or ("ShuffledHashJoin" in plan_default), join_default[:3]
assert "ShuffledHashJoin" in plan_shj, join_shj[:3]
print("\nQ8 OK — broadcast when small, shuffled join when big, SHUFFLE_HASH hint forced SHJ")

print("""
[MECHANICS]
SMJ: sort(build) + sort(probe) + merge walk. Work: O(B log B) + O(P log P) + one scan each.
SHJ: hash build side into per-partition hash tables (RAM), then STREAM the probe side. No sorts.

[WHY CLASSIC SPARK DEFAULTS TO SMJ (and modern runtimes sometimes don't)]
SHJ must hold the ENTIRE build-side partition in memory per task. If build_per_partition >
execution memory -> OOM (the hash table cannot spill in classic SHJ). SMJ is spill-safe:
both phases spill to disk. Classic Spark refuses to gamble OOM on a plan that only MIGHT
fit - so it plans SMJ and lets you opt into SHJ. Modern engines (Photon / AQE) know real
shuffle sizes at runtime and will pick SHJ THEMSELVES when it is provably safe - exactly
the sizing logic below, which is why you should be able to recite it.

[THE SIZING RULE]
build_per_partition ~= build_bytes / shuffle_partitions. With 2,000 partitions, a 2GB build
side = ~1MB/partition -> trivially fits. With 20 partitions -> 100MB/partition -> still OK
on a 16g heap, but at 2GB/partition you are back to OOM risk. SHJ wins when: partitions are
plentiful, the probe side is huge (sorting 4TB is expensive!), build per-partition is small.

[HOW TO FORCE IT]
1. spark.sql.join.preferSortMergeJoin=false     (cluster-wide, affects ALL joins - risky)
2. df.hint('SHUFFLE_HASH')                      (per-query, preferred)
3. If a side is ALREADY sorted (e.g. bucketed on the join key) SMJ skips the sort -> keep SMJ.
4. AQE can also rewrite plans at runtime once real shuffle sizes are known.

[ANSWER TO THE INTERVIEWER] With 2,000 partitions and a 2GB build side, the peer is right:
SHJ skips sorting the 4TB probe side and ~1MB build partitions fit safely in memory. Force it
per-query with the hint, then confirm zero spill in the Spark UI.
""")

## Q9 — ETL: Zero-Downtime Schema Migration — Expand & Contract [ADVANCED]

**Interviewer (Stripe):** Split `full_name` into `first_name` / `last_name` on a gold table consumed by **15 dashboards and 3 downstream pipelines**. Zero downtime, zero breakage, rollback-safe.

1. Explain **expand-contract**: expand (add new columns, dual-write), migrate (backfill history), contract (drop old column). Why can't you just `ALTER TABLE ... RENAME`?
2. How does a **blue-green view swap** make the cutover atomic for consumers? Show the sequence.
3. Why must the **contract phase wait** for consumer verification? How do you detect consumers still reading `full_name` (query history / access logs) before dropping it?

In [0]:
# Solution Q9 — expand-contract with blue-green view swap (runnable on a volume-backed path)
from pyspark.sql import functions as F

tbl = "/Volumes/workspace/default/etl_test_volume/d19_q9/customers"
dbutils.fs.rm(tbl, True)  # idempotent re-run

# --- original table (old world) ---
spark.createDataFrame([(1, "Grace Hopper", "COBOL Corp", "2026-10-01"),
                       (2, "Alan Turing", "Enigma Ltd", "2026-10-05")],
                      ["id", "full_name", "company", "signup"]) \
     .write.format("delta").save(tbl)

# --- PHASE 1: EXPAND (non-breaking: additive only) ---
spark.sql(f"ALTER TABLE delta.`{tbl}` ADD COLUMNS (first_name STRING, last_name STRING)")

# --- PHASE 2: DUAL-WRITE (writer populates BOTH old and new columns) ---
new_rows = spark.createDataFrame([(3, "Ada Lovelace", "Analytical Eng", "2026-10-28")],
                                 ["id", "full_name", "company", "signup"])
new_rows = (new_rows
    .withColumn("first_name", F.split("full_name", " ").getItem(0))
    .withColumn("last_name", F.element_at(F.split("full_name", " "), -1)))
new_rows.write.format("delta").mode("append").save(tbl)

# --- PHASE 3: MIGRATE / BACKFILL history ---
spark.read.format("delta").load(tbl).createOrReplaceTempView("cust_v")
spark.sql(f"""
MERGE INTO delta.`{tbl}` t
USING (
  SELECT id,
         split(full_name, ' ')[0]               AS first_name,
         element_at(split(full_name, ' '), -1)  AS last_name
  FROM cust_v WHERE first_name IS NULL
) s ON t.id = s.id
WHEN MATCHED THEN UPDATE SET t.first_name = s.first_name, t.last_name = s.last_name
""")

# --- PHASE 4: BLUE-GREEN VIEW SWAP (atomic for every consumer) ---
# Consumers read the VIEW, never the base table. The flip is one CREATE OR REPLACE -> atomic.
spark.sql(f"""CREATE OR REPLACE VIEW customers_v2 AS
             SELECT id, first_name, last_name, company, signup, full_name
             FROM delta.`{tbl}`""")

# --- PHASE 5: CONTRACT (only after query logs show nobody reads full_name) ---
chk = spark.sql(f"SELECT id, first_name, last_name FROM delta.`{tbl}` ORDER BY id")
chk.show()
assert chk.filter("first_name IS NULL OR last_name IS NULL").count() == 0  # backfill complete
assert chk.count() == 3

print("""
[CONTRACT CHECKLIST]
- Scan query history / UC audit logs for 'full_name' reads over the last N days; empty -> safe
  to DROP COLUMN. Non-empty -> keep deriving full_name in the view for stragglers.
- Rollback at any phase is trivial: additive columns are ignored by old readers until the swap;
  the view flip is reversible in one statement. ALTER ... RENAME would have broken every
  consumer INSTANTLY and irreversibly - exactly what expand-contract avoids.
""")
print("Q9 OK — expand-contract executed end-to-end with atomic view swap")
spark.sql("DROP VIEW IF EXISTS customers_v2")
dbutils.fs.rm(tbl, True)  # clean up demo artifacts

## Q10 — ETL: DEEP CLONE — Backup, DR & Environment Seeding [ADVANCED]

**Interviewer (Adobe):** Before a risky reprocessing job, you must copy a production gold table into a staging environment. The table is 5 TB with 60 days of history. Compare your options and write the workflow.

1. **Deep clone vs shallow clone**: what is physically copied (data files vs metadata)? What does a shallow clone cost, and when is it a trap (source deleted/vacuumed)?
2. Write the workflow: clone -> run the risky job on the clone -> validate parity -> **refresh** the clone from an updated source. What is the **clone-on-write** behavior when the source keeps changing?
3. Position deep clone against `RESTORE`/time travel and `VACUUM` retention in an overall DR strategy — what protects against what?

In [0]:
# Solution Q10 — shallow vs deep clone, isolation check, incremental refresh (managed tables)
SRC, DEEP, SHALLOW = ("workspace.default.d19_q10_prod",
                      "workspace.default.d19_q10_deep",
                      "workspace.default.d19_q10_shallow")
for t in (SRC, DEEP, SHALLOW):
    spark.sql(f"DROP TABLE IF EXISTS {t}")   # idempotent re-run

# --- the production gold table (managed) ---
spark.sql(f"""CREATE OR REPLACE TABLE {SRC} AS
             SELECT * FROM VALUES
               (1, '2026-10-01', 120.0),
               (2, '2026-10-02',  80.0) AS t(order_id, dt, amount)""")

# --- SHALLOW clone: metadata only; data files are SHARED with the source ---
spark.sql(f"CREATE OR REPLACE TABLE {SHALLOW} SHALLOW CLONE {SRC}")

# --- DEEP clone: physical copy of data files + full version history ---
spark.sql(f"CREATE OR REPLACE TABLE {DEEP} DEEP CLONE {SRC}")

# --- run the 'risky job' on the CLONE, not on prod ---
spark.sql(f"INSERT INTO {DEEP} VALUES (3, '2026-10-03', 42.0)")

# --- parity validation: prod untouched, clone diverged (isolation worked) ---
prod_rows    = spark.sql(f"SELECT count(*) FROM {SRC}").first()[0]
deep_rows    = spark.sql(f"SELECT count(*) FROM {DEEP}").first()[0]
shallow_rows = spark.sql(f"SELECT count(*) FROM {SHALLOW}").first()[0]
assert (prod_rows, deep_rows, shallow_rows) == (2, 3, 2), (prod_rows, deep_rows, shallow_rows)
print(f"prod={prod_rows}, deep_clone={deep_rows}, shallow_clone={shallow_rows} -> risky job fully isolated")

# --- source keeps changing; incremental refresh: re-issue the same DEEP CLONE ---
spark.sql(f"INSERT INTO {SRC} VALUES (4, '2026-10-04', 9.0)")
spark.sql(f"CREATE OR REPLACE TABLE {DEEP} DEEP CLONE {SRC}")
assert spark.sql(f"SELECT count(*) FROM {DEEP}").first()[0] == 3  # refreshed to source state

print("""
[WHAT TO SAY OUT LOUD]
- SHALLOW clone copies metadata/transaction log only; files SHARED with the source. Instant,
  free-ish. TRAP: if the source is vacuumed or dropped, the shallow clone's old versions
  break -> never a long-lived backup. Fine for dev/testing at prod scale.
- DEEP clone copies data files + full version history. Cost = storage. Survives source VACUUM.
- On Unity Catalog, clones are MANAGED tables (the old path-based LOCATION form needs a
  cloud-scheme URI and is legacy; shallow clone in particular only supports managed tables).
- Clone-on-write: writes to either side create NEW files; the shared prefix is never mutated.
- Refresh: re-issuing the same DEEP CLONE copies only files added since the last clone.
- DR matrix: time travel/RESTORE -> logical mistakes on the SAME table (bounded by VACUUM
  retention, default 7 days); DEEP CLONE -> independent copy immune to source vacuum/drop;
  cross-region replication -> site failure. Time travel is the oops layer, clone is the DR layer.
""")
for t in (SHALLOW, DEEP, SRC):
    spark.sql(f"DROP TABLE IF EXISTS {t}")   # clean up demo artifacts
print("Q10 OK — shallow vs deep cloned, risky job isolated, clone refreshed incrementally")

## Q11 — ETL: Zero-Loss Streaming Upgrades — Checkpoints & State [HARD]

**Interviewer (Uber):** A 24/7 Kafka→Delta pipeline (deduplication + running aggregates, stateful, 400M keys) must move to a new Spark version **with a changed aggregation**. No event may be lost or double-counted. Walk me through the upgrade.

1. What exactly lives in a **checkpoint** (offsets, commits, state store)? Why does changing the query shape break it — what error would you hit?

2. Give the **safe upgrade runbook**: drain the query with `Trigger.AvailableNow`, validate final state, start the new version on a **fresh checkpoint**, replay the overlap window, verify parity with counts/checksums. Which state-store configs matter (RocksDB provider, changelog checkpointing)?

3. Classify changes: **checkpoint-compatible** (adding a column) vs **incompatible** (changing groupBy keys / aggregation logic / watermark column). How do you migrate state when the change is incompatible?

In [0]:
# Solution Q11 — safe-upgrade runbook (playbook + the drain pattern in code)

RUNBOOK = {
  "1. DRAIN": "Set Trigger.AvailableNow (processes ALL pending source data as finite batches,
      then stops cleanly). awaitTermination. Offsets fully committed -> nothing in flight.
      This is the ONLY clean way to stop a stateful query - never kill -9 it.",
  "2. FREEZE A SNAPSHOT": "Backup the Delta sink + the checkpoint dir (offsets/, commits/,
      state/) + record the final watermark and last committed offset per Kafka partition.",
  "3. PREPARE THE NEW QUERY": "New Spark version, new aggregation. Point it at a FRESH
      checkpoint location - NEVER reuse the old checkpoint across incompatible query changes.",
  "4. RESEED THE OVERLAP": "Replay from the last committed offset (set startingOffsets for
      Kafka; idempotent reprocess of the window for file sources). The new aggregation
      rebuilds state from scratch over the overlap window.",
  "5. DEDUPE THE SINK": "Writes must be idempotent: MERGE on (partition, offset) keys or
      REPLACEWHERE the overlap window so replayed events do NOT double-write.",
  "6. VERIFY PARITY BEFORE SWITCH": "counts, sums, row-level checksums old vs new across the
      overlap; only then repoint consumers. Keep the old pipeline paused for instant rollback.",
}
for step, text in RUNBOOK.items():
    print(step, "->", text.replace("\n      ", " "), "\n")

# --- the drain pattern (the code they want to SEE) ---
drain_snippet = '''
# OLD pipeline: switch trigger to AvailableNow to drain instead of killing it
q = (streaming_df.writeStream
       .format("delta")
       .option("checkpointLocation", ckpt)
       .trigger(availableNow=True)     # <-- finite: drains, commits, stops
       .start())
q.awaitTermination()                  # clean stop; offsets committed

# NEW pipeline: fresh checkpoint + RocksDB state store + changelog checkpointing
spark.conf.set("spark.sql.streaming.stateStore.providerClass",
               "com.databricks.sql.streaming.state.RocksDBStateStoreProvider")
spark.conf.set("spark.sql.streaming.stateStore.rocksdb.changelogCheckpointing.enabled",
               "true")
q2 = (new_streaming_df.writeStream
        .format("delta")
        .option("checkpointLocation", NEW_CKPT)  # fresh - never reuse across incompatible changes
        .trigger(processingTime="30 seconds")
        .start())
'''
print(drain_snippet)

COMPAT = {
  "checkpoint-COMPATIBLE (may reuse checkpoint)": [
    "adding a new output column / sink field",
    "changing trigger cadence",
    "map-only output changes that don't touch state keys",
  ],
  "INCOMPATIBLE (fresh checkpoint + reseed required)": [
    "changing groupBy keys / flatMapGroupsWithState key or state schema",
    "changing aggregation logic",
    "changing the watermark column or delay",
    "Spark version jumps that change state-store format versions",
  ],
  "migrating state when incompatible": [
    "export state to a Delta table (foreachBatch flush), transform it, then load it as the
     initial seed for the new query - rebuild state, never port it",
  ],
}
for k, v in COMPAT.items():
    print(k + ":")
    for x in v:
        print("   -", x)
print("\nQ11 OK — drain pattern + fresh-checkpoint rule + compatibility matrix")

## Q12 — Data Modeling: One Big Table vs Star Schema at BI Scale [ADVANCED]

**Interviewer (Airbnb):** Analysts complain their dashboards are slow because of 6 joins across a classic star schema. Leadership asks: *why don't we just pre-join everything into one big denormalized table (OBT)?*

1. Compare the two: query simplicity & scan efficiency vs storage bloat, **SCD re-computation cost** (a dimension update rewrites how many fact rows?), maintenance, and flexibility.
2. Give a **decision framework**: usage pattern (fixed dashboard vs ad-hoc), table sizes, dimension churn rate, BI tool constraints.
3. How does the medallion **gold layer** let you serve BOTH patterns? When does OBT itself become the bottleneck (rebuild cost, late-arriving dimension fixes, dimension fan-out)?

In [0]:
# Solution Q12 — runnable comparison + the SCD-rebuild cost that decides it
from pyspark.sql import functions as F

# --- tiny star schema ---
dim_customer = spark.createDataFrame([
    (10, "Alice", "gold",   "2026-01-01", "9999-12-31"),
    (20, "Bob",   "silver", "2026-01-01", "9999-12-31"),
], ["customer_sk", "name", "tier", "valid_from", "valid_to"])

dim_product = spark.createDataFrame([
    (100, "Widget"), (200, "Gadget"),
], ["product_sk", "product_name"])

fact = spark.createDataFrame([
    ("2026-10-01", 10, 100, 2),
    ("2026-10-01", 20, 200, 1),
    ("2026-10-02", 10, 200, 5),
], ["dt", "customer_sk", "product_sk", "qty"])

star = fact.join(dim_customer, "customer_sk").join(dim_product, "product_sk")
star.select("dt", "name", "tier", "product_name", "qty").show()

# --- OBT = the same thing, materialized ---
obt = star.select("dt", "name", "tier", "product_name", "qty")
assert obt.count() == star.count() == 3

# --- THE decisive demo: dimension churn cost ---
# Alice is re-tiered (SCD2 update) -> in a STAR you write 1 dim row.
# In an OBT you must REWRITE every fact row carrying Alice's attributes.
rows_alice = obt.filter("name='Alice'").count()
print(f"Alice re-tiered -> star: 1 new dim row ({len(dim_customer.columns)} cols). "
      f"OBT: rewrite {rows_alice} fact rows ({len(obt.columns)} cols each, measures included).")

share = rows_alice / obt.count()
print(f"At this scale that's {share:.0%} of the table; at 1B facts and 10M customers a single
hot dimension attribute change rewrites its FULL share of the OBT - every time.")

print("""
[DECISION FRAMEWORK]
- Fixed dashboards, few grains, high query volume, analysts who want SQL-simple? -> OBT marts.
- Ad-hoc analysis, many grains, frequent dimension corrections? -> star schema joins.
- BI tool joins poorly or charges per table scanned? -> OBT.
- High dimension churn (tier/status/score)? -> star, or OBT with current-only dimensions.

[GOLD LAYER = BOTH]
A conformed star for flexibility + a handful of OBT marts for hot dashboards, refreshed
incrementally (MERGE on changed keys, not full rebuilds). Late-arriving dimension fixes then
touch the star + a re-MERGE of the affected mart slices - never a full OBT rebuild.
""")
print("Q12 OK — parity verified; churn cost quantified")

## Q13 — Data Modeling: Ragged (Unbalanced) Hierarchies [ADVANCED]

**Interviewer (Walmart):** Our supplier hierarchy is **ragged** — some branches are 3 levels deep, others 7. Finance must be able to roll up spend at **every** level, and the BI tool cannot do recursive joins.

1. Why do parent-child schemas fail BI rollups? (recursive CTE per query, ragged paths, no level columns to GROUP BY)
2. Design the **level-flattened dimension**: fixed `lvl1..lvlN` columns + a `depth` column. How do you handle nodes at shallower depth — pad with `UNKNOWN`/`Not Applicable`, or repeat the leaf member? Which is correct for aggregation, and why?
3. Trade-offs: fixed depth limits (adding an 8th level?), refresh cost, and how you support "drill any level" dashboards.

In [0]:
# Solution Q13 — parent-child -> level-flattened dimension (runnable, depth 5)
from pyspark.sql import functions as F, Window

# Ragged parent-child: 'Global' is depth 1; branches end at different depths.
edges = spark.createDataFrame([
    ("Global",       None),
    ("Home",        "Global"),
    ("Fashion",     "Global"),     # depth-2 branch (ragged!)
    ("Electronics", "Home"),
    ("Appliances",  "Home"),
    ("TVs",         "Electronics"),
    ("Laptops",     "Electronics"),
    ("Washers",     "Appliances"),
    ("OLED",        "TVs"),        # depth 5 (leaf)
    ("QLED",        "TVs"),        # depth 5 (leaf)
    ("Gaming",      "Laptops"),    # depth 5 (leaf)
    ("Front-load",  "Washers"),    # depth 5 (leaf)
], ["node", "parent"])

spend = spark.createDataFrame([
    ("OLED", 1000.0), ("QLED", 800.0), ("Gaming", 600.0), ("Front-load", 400.0),
], ["leaf", "spend_usd"])

MAX_LEVELS = 5

def build_flattened(edges, max_levels=5):
    """Walk each node up to the root (max_levels hops), then expose lvl1..lvlN columns."""
    result = edges.select(F.col("node").alias("node_id"), "node", "parent",
                          F.array("node").alias("path"), F.lit(1).alias("depth"))
    for _ in range(max_levels - 1):
        nxt = (result
               .join(edges.select(F.col("node").alias("pn"), F.col("parent").alias("pp")),
                     result["parent"] == F.col("pn"), "left")
               .select("node_id", "node",
                       F.col("pp").alias("parent"),
                       F.when(F.col("pn").isNotNull(), F.concat(F.array("pn"), "path"))
                            .otherwise(F.col("path")).alias("path"),
                       F.when(F.col("pn").isNotNull(), F.col("depth") + 1)
                            .otherwise(F.col("depth")).alias("depth")))
        result = result.unionByName(nxt)
    # one row per node: the row with its FULL root-to-node path (longest path)
    w = Window.partitionBy("node_id").orderBy(F.size("path").desc())
    full = result.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn", "node", "parent")
    # explode the path into fixed level columns; pad shallow branches explicitly
    # (get() is 0-BASED and tolerates out-of-bounds by returning NULL; element_at
    #  is 1-based but THROWS on out-of-bounds - use get() here)
    for i in range(max_levels):
        full = full.withColumn(f"lvl{i+1}", F.coalesce(F.get("path", i), F.lit("Not Applicable")))
    return full.select("node_id", *[f"lvl{i+1}" for i in range(max_levels)], "depth")

dim = build_flattened(edges, MAX_LEVELS)
dim.filter("depth = 5").orderBy("node_id").show(truncate=False)
dim.filter("node_id IN ('Global','Fashion','TVs')").show(truncate=False)

# --- rollup at ANY level is now a plain GROUP BY (what the BI tool needed) ---
rollup = (dim.join(spend, dim.node_id == spend.leaf)
          .groupBy("lvl4").agg(F.sum("spend_usd").alias("spend")))
rollup.orderBy("lvl4").show()

tv_spend = rollup.filter("lvl4='TVs'").first()["spend"]
assert tv_spend == 1800.0, tv_spend
assert dim.count() == 12
assert dim.filter("depth = 5").count() == 4
# shallow branches are padded, not clipped
fash = dim.filter("node_id='Fashion'").first()
assert fash["lvl1"] == "Global" and fash["lvl2"] == "Fashion" and fash["lvl3"] == "Not Applicable"
oled = dim.filter("node_id='OLED'").first()
assert [oled[f"lvl{i}"] for i in range(1, 6)] == ["Global", "Home", "Electronics", "TVs", "OLED"]

print("""
[PADDING CHOICE - SAY THIS OUT LOUD]
- 'Not Applicable' (UNKNOWN member): spend under a depth-2 branch like Fashion does NOT
  silently re-appear under a fake lvl3/lvl4/lvl5; drilling shows an explicit unknown node.
  This is the Kimball recommendation - aggregation stays CORRECT and member counts stay honest.
- Repeating the leaf (Fashion under Fashion under Fashion): rollups stay numerically right,
  but drill paths look wrong and distinct-member counts break. Only if the BI tool cannot
  handle the N/A member.
- Fixed depth CLIPS anything deeper (a 6th level here would be cut). Mitigate:
  over-provision levels (say 8), monitor MAX(depth) at every refresh, and alert before
  the ceiling is hit.
- Refresh cost: one O(levels) self-join pass per refresh - cheap; the ORIGINAL recursive
  CTE per dashboard query was the expensive part.
""")
print("Q13 OK — ragged hierarchy flattened to 5 fixed levels; rollup at any level verified")

## Q14 — Snowflake: Data Sharing Architecture [ADVANCED]

**Interviewer (Salesforce):** You must monetize 200 tables to 40 external customer accounts **without copying data and without paying for their compute**. Explain Snowflake data sharing end-to-end.

1. How does a share actually work — metadata pointers, consumer account, same cloud/region? What is **NOT** copied (no data movement, no storage charges for the provider's share)?
2. Compare **direct sharing** vs **listings** (Snowflake Marketplace) vs replication for cross-region consumers. What is a **secure share**?
3. Governance: what happens to **row access policies and masking policies** when data is shared? Who is billed for queries? Which objects can be added to a share?

In [0]:
# Solution Q14 — the data-sharing talk track with the SQL you'd write

provider_sql = """
-- 1) create the share and add objects (tables, and SECURE VIEWS over raw data)
CREATE OR REPLACE SHARE customer_metrics_share;
ALTER SHARE customer_metrics_share ADD TO DATABASE analytics.shared_data;
GRANT USAGE ON DATABASE analytics TO SHARE customer_metrics_share;
GRANT USAGE ON SCHEMA analytics.shared_data TO SHARE customer_metrics_share;
ALTER SHARE customer_metrics_share ADD TABLE analytics.shared_data.daily_revenue;
-- expose a governed subset instead of the raw table:
ALTER SHARE customer_metrics_share ADD SECURE VIEW analytics.shared_data.v_revenue_per_tenant;

-- 2) grant a consumer account (their account locator)
ALTER SHARE customer_metrics_share ADD ACCOUNTS = <consumer_account_locator>;

-- 3) consumer side: the share becomes a database - zero data copied, zero storage bill
CREATE DATABASE revenue_feed FROM SHARE <provider_org>.customer_metrics_share;
SELECT * FROM revenue_feed.shared_data.daily_revenue;
"""
print(provider_sql)

TALK_TRACK = {
  "how it works":
    "The share is METADATA: pointers to micro-partitions in the provider's cloud storage.
    No data is copied. The consumer creates a database FROM the share and queries it with
    THEIR OWN virtual warehouse (consumer pays compute, provider pays storage once).",
  "region requirement":
    "The consumer account must be in the same cloud + region as the shared tables.
    Cross-region needs database REPLICATION (provider replicates into the consumer's
    region -> NOW there are storage copies + replication credits).",
  "secure view in a share":
    "Share a SECURE VIEW, not the base table: hides the definition, lets you filter rows
    per tenant. The consumer never sees the raw layer.",
  "governance travels with the data":
    "Row access policies and masking policies attached to shared objects are enforced for
    the consumer too - policy functions evaluate in the PROVIDER's account context.
    Gotcha to name: a policy branching on the CONSUMER's role breaks (unknown role) -
    key it on CURRENT_ACCOUNT() / an account mapping table, and test it.",
  "listings / marketplace vs direct":
    "Direct share = 1:1, you manage the account list. Listing = productized share
    (auto-fulfill, usage terms, pricing). Same sharing mechanics underneath.",
  "what can be shared":
    "Tables, views, secure views, secure UDFs, and schemas. NOT: dynamic tables, hybrid
    tables, stages, external tables - partial shares must stick to classic shareable objects.",
  "cost model to state":
    "Provider: storage only (already paying). Consumer: compute for every query against the
    imported database. Replication adds provider storage + transfer credits.",
}
for k, v in TALK_TRACK.items():
    print(f"[{k.upper()}]\n  {v}\n")
print("Q14 OK — share mechanics, governance, and cost model articulated with SQL")

## Q15 — Snowflake: Event Tables & Snowflake Logging [ADVANCED]

**Interviewer (Netflix):** A Snowpark stored procedure fails intermittently for **one tenant only**. You need logs and exceptions from inside Snowflake code — `print` goes nowhere.

1. What is an **event table**? How does it differ from a normal table (append-only, structured logging schema, lifecycle-managed retention)?
2. Use **Snowflake Logging** inside a Snowpark UDF/procedure: `logging.info/error`, the `TRACE_ON` level, and `EXCEPTION` events. Where do records land — name the key columns you query (`TIMESTAMP`, `RECORD`, `EVENT_NAME`, `SEVERITY`)?
3. Cost and control: how do log levels throttle volume, what's the retention lifecycle, and why is an event table better than a logging table you hand-roll?

In [0]:
# Solution Q15 — event table setup + Snowpark logging + the query you debug with

setup_sql = """
-- one event table per account (convention: <org>_events)
CREATE EVENT TABLE IF NOT EXISTS my_account_events;
-- route this user's logs/exceptions/traces to it (account or user level)
ALTER USER ANALYST_JANE SET EVENT_TABLE = mydb.events.my_account_events;

-- throttle what gets captured (default INFO; TRACE is a firehose - use sparingly)
ALTER SESSION SET LOG_LEVEL = INFO;
ALTER SESSION SET TRACE_LEVEL = ON_EVENT;   -- TRACE_ON(...) lines emit only when set
"""
print(setup_sql)

snowpark_code = '''
-- inside a Snowpark Python stored procedure
CREATE OR REPLACE PROCEDURE mydb.ops.process_tenant(tenant STRING)
  RETURNS STRING
  LANGUAGE PYTHON
  RUNTIME_VERSION = '3.10'
  PACKAGES = ('snowflake-snowpark-python')
  HANDLER = 'run'
AS $$
import logging

def run(session, tenant):
    log = logging.getLogger("tenant_proc")
    log.info("process_tenant START", extra={"tenant": tenant})
    try:
        rows = session.sql("SELECT COUNT(*) FROM facts WHERE tenant_id = ?", params=[tenant]).collect()
        n = rows[0][0]
        log.debug("row count fetched", extra={"tenant": tenant, "rows": n})
        log.info("process_tenant DONE", extra={"tenant": tenant, "rows": n})
        return str(n)
    except Exception:
        log.error("process_tenant FAILED", extra={"tenant": tenant})
        raise          # EXCEPT events also land in the event table automatically
$$;
'''
print(snowpark_code)

query_sql = """
-- the 3 AM query: this tenant's last failures with full detail
SELECT TIMESTAMP, EVENT_NAME, SEVERITY, RECORD, RESOURCE_ATTRIBUTES, EXCEPTION
FROM my_account_events
WHERE TIMESTAMP > DATEADD(hour, -24, CURRENT_TIMESTAMP())
  AND RESOURCE_ATTRIBUTES["snow.database.name"]:"string" = 'MYDB'
  AND RECORD["tenant"]:"string" = 'ACME'
  AND SEVERITY >= 'ERROR'
ORDER BY TIMESTAMP DESC;
"""
print(query_sql)

NOTES = {
  "columns you must be able to name":
    "TIMESTAMP, EVENT_NAME (LOG | EXCEPT | TRACE | SPAN), SEVERITY, RECORD (the structured
     key/value payload from extra={...}), RESOURCE_ATTRIBUTES (db/schema/user/object
     context), EXCEPTION (stack trace for EXCEPT events), RECORD_ATTRIBUTES",
  "lifecycle/retention":
    "The event table is backed by lifecycle-managed storage: Snowflake moves old partitions
    to cheaper storage and expires them automatically. It is APPEND-ONLY - you cannot
    DELETE from it; control volume with log levels instead.",
  "log levels as cost control":
    "ERROR / WARN / INFO / DEBUG plus TRACE. DEBUG and TRACE in steady state will drown the
     table and the bill - enable them per-session or with TRACE_LEVEL = ON_EVENT.",
  "vs hand-rolled logging table":
    "Native: automatic context (user, object, role), zero ETL, lifecycle retention, and it
    works inside UDFs/procs where you have no file system. Hand-rolled: full control, but
    you rebuild all of that - and it usually can't log from inside UDF execution anyway.",
}
for k, v in NOTES.items():
    print(f"[{k}]\n  {v}\n")
print("Q15 OK — event table + Snowpark logging + the debug query")

## Q16 — Monitoring & Orchestration: For-Each Fan-Out with Task Values [ADVANCED]

**Interviewer (Databricks):** A nightly job must run the same ingest per-region (40 regions) **in parallel**, then aggregate. Design it with **For Each tasks** and **task values**.

1. How does a For-Each task fan out? The **mapping task** (returns the JSON array) vs the **inner task** (runs per element) vs the **shared task** choice; concurrency controls; what happens when one iteration fails?
2. **Task values**: `dbutils.jobs.taskValues.set(...)` in the inner task, `dbutils.jobs.taskValues.get(...)` by `taskKey` in the fan-in — show the fan-out/fan-in wire-up.
3. Failure & repair semantics: do sibling iterations continue when one fails? How do you repair ONLY the failed regions on rerun (per-iteration retries, repair runs)?

In [0]:
# Solution Q16 — For-Each + task values, simulated locally + the real wiring to recite
import json

REGIONS = ["us-east-1", "us-west-2", "eu-west-1", "ap-south-1"]

# ---------- (1) MAPPING TASK notebook: returns the array the For-Each consumes ----------
mapping_task_code = '''
import json

# regions needing a refresh, discovered from a control table
regions = [r.region for r in spark.sql(
    "SELECT DISTINCT region FROM ops.refresh_regions WHERE to_run = true").collect()]
dbutils.notebook.exit(json.dumps(regions))   # <- the For-Each task consumes this array
'''
print("MAPPING TASK:\n", mapping_task_code)

# ---------- (2) INNER TASK notebook: parameterized per iteration; publishes a task value ----------
inner_task_code = '''
# receives the iteration element as the parameter/widget named 'region'
region = dbutils.widgets.get("region")

df = ingest(region)                        # the actual work
dbutils.jobs.taskValues.set(key="rows_ingested", value=df.count())   # for the fan-in
if df.count() == 0:
    raise ValueError(f"zero rows for {region} - fail THIS iteration loudly")
'''
print("INNER TASK:\n", inner_task_code)

# ---------- (3) FAN-IN task: reads each iteration's task values by task key ----------
fan_in_code = '''
total, failed = 0, []
for region in REGIONS:
    try:
        n = dbutils.jobs.taskValues.get(taskKey="ingest_region", key="rows_ingested",
                                        default=None, debugValue=0)
        total += n
    except Exception:
        failed.append(region)
dbutils.notebook.exit(json.dumps({"total": total, "failed": failed}))
'''
print("FAN-IN TASK:\n", fan_in_code)

# ---------- local simulation of the whole flow (proves the wiring logic) ----------
def ingest(region):
    return {"us-east-1": 1200, "us-west-2": 800, "eu-west-1": 950, "ap-south-1": 0}[region]

results, failed = {}, []
for region in REGIONS:                      # the For-Each body, once per element
    n = ingest(region)
    results[region] = n
    if n == 0:
        failed.append(region)                # this iteration fails; siblings DON'T

total = sum(v for k, v in results.items() if k not in failed)
print(f"\nsimulated fan-out: {json.dumps(results)} | total={total} | failed={failed}")
assert failed == ["ap-south-1"] and total == 2950

print("""
[SEMANTICS TO SAY OUT LOUD]
- Concurrency is set on the For-Each task (e.g. max 10 parallel) to protect the source.
- One iteration failing does NOT kill sibling iterations - each is its own task run. The
  For-Each task reports failure; route the fan-in with 'Run If: ALL_DONE' when you want a
  post-mortem aggregate even on partial failure.
- Repair: on rerun/repair-run, only FAILED iterations re-execute (succeeded ones are skipped)
  - pair with idempotent MERGE writes so a partial rerun is safe.
- Shared task vs inner task: shared = one task definition used by other jobs too; inner =
  private to this job. Task values from a shared task need the task key per iteration.
- Keep the mapping task tiny and deterministic - the entire fan-out depends on it.
""")
print("Q16 OK — fan-out/fan-in wired with task values; failure semantics verified")

## Q17 — Monitoring & Testing: Unit-Testing Data Pipelines (pytest + chispa) [ADVANCED]

**Interviewer (Spotify):** A currency-conversion transform silently rounds wrong for one locale, and it shipped to production. How do you add tests so it can never ship again?

1. What CAN be unit-tested in a data pipeline (pure transforms, UDFs, expectation logic) vs what needs integration tests (sources, sinks, cluster configs)? Sketch the **test pyramid** for data.
2. Write pytest cases for a PySpark transform using **chispa** (`assert_df_equality`, `assert_schema_equality`) — including the locale/rounding case that was broken.
3. Where do these tests run (CI pre-merge, `%pip install pytest chispa`), and what's the golden-dataset / snapshot strategy for regressions?

In [0]:
# Solution Q17 — pure transform + chispa/pytest test harness
# (in a real repo: %pip install pytest chispa --quiet, then `pytest tests/` in CI)
from pyspark.sql import functions as F, types as T
from decimal import Decimal

try:
    import chispa
    CHISPA = True
except ImportError:
    CHISPA = False
    print("[note] chispa not installed here -> row-level asserts carry the same semantics")

# ---------- the transform under test (historical bug FIXED and now locked in) ----------
RATE_SCHEMA = T.StructType([T.StructField("locale", T.StringType()),
                            T.StructField("rate", T.DoubleType())])

def convert_currency(df, rates):
    """amount_cents EUR -> local currency, HALF_UP rounded to 2 decimals.
    The old bug: plain float division + Python round() (banker's rounding) mis-rounded
    x.x5 cases - e.g. 12.085 -> 12.08 instead of 12.09."""
    r = spark.createDataFrame(rates, RATE_SCHEMA)
    return (df.join(r, "locale")
            .withColumn("amount_local",
                        F.expr("CAST(round(amount_cents / 100.0 * rate, 2) AS DECIMAL(18,2))"))
            .select("order_id", "locale", "amount_cents", "rate", "amount_local"))

EXPECTED_SCHEMA = "order_id BIGINT, locale STRING, amount_cents BIGINT, rate DOUBLE, amount_local DECIMAL(18,2)"

# ---------- tests (pytest collects these as test_* functions) ----------
def test_half_up_rounding_and_schema():
    df = spark.createDataFrame([(1, "en-US", 1099),   # 10.99 * 1.10 = 12.089 -> 12.09
                                (2, "ja-JP", 250)],   # 2.50 * 161.80 = 404.50
                               ["order_id", "locale", "amount_cents"])
    out = convert_currency(df, [("en-US", 1.10), ("ja-JP", 161.80)])
    expected = spark.createDataFrame(
        [(1, "en-US", 1099, 1.10, Decimal("12.09")),
         (2, "ja-JP", 250, 161.80, Decimal("404.50"))], EXPECTED_SCHEMA)

    if CHISPA:
        chispa.assert_df_equality(out, expected, ignore_row_order=True, ignore_nullable=True)
        chispa.assert_schema_equality(out.schema, expected.schema, ignore_nullable=True)

    rows = {r["order_id"]: r["amount_local"] for r in out.collect()}
    assert str(rows[1]) == "12.09"      # the HALF_UP case that used to ship as 12.08
    assert str(rows[2]) == "404.50"


def test_no_rate_means_no_row():
    df = spark.createDataFrame([(9, "xx-XX", 100)], ["order_id", "locale", "amount_cents"])
    out = convert_currency(df, [("en-US", 1.10)])
    assert out.count() == 0             # inner join drops unconvertible orders - document it

test_half_up_rounding_and_schema()
test_no_rate_means_no_row()

print("""
[THE TEST PYRAMID FOR DATA - SAY THIS OUT LOUD]
1. Unit (fast, every PR): pure transforms/UDFs on tiny DataFrames - chispa row/schema
   asserts. Hundreds run in minutes.
2. Integration (nightly): real sources/sinks, Delta commits, MERGE logic - a few, against
   staging catalogs.
3. Golden datasets (regression): curated input->expected snapshots; compare schema hash +
   row checksums so unintended drift FAILS the build instead of shipping.
CI runs 1 on every PR; 2+3 on schedule. Tests live with the notebook code in the repo.
""")
print("Q17 OK — HALF_UP rounding regression locked in; schema + drop semantics covered")

## Q18 — Monitoring: Design a Data Observability Stack for 500 Pipelines [HARD]

**Interviewer (Netflix):** Our platform runs 500 pipelines and 3,000 tables, with SLAs owed to external customers. Design the **observability layer** end-to-end — metrics, storage, alerting, on-call, and cost.

1. **Metric hierarchy**: pipeline health (job/task-level) vs data health (freshness, volume, schema, quality) vs cost. Which system tables back each layer?
2. **Architecture**: where metrics land (metrics tables + quality monitors), how alerts route (severity -> channel), and how they link to runbooks. Why alert on **symptoms (SLA breach)** rather than causes (job failed)?
3. **Anti-noise & operations**: alert budgets per team, dedup/flapping control, weekly review. How do you keep 500 pipelines from generating 5,000 alerts a week?

In [0]:
# Solution Q18 — the layered observability blueprint (design question -> design answer)

BLUEPRINT = {
  "L1 PIPELINE HEALTH (is the machinery running?)": """
    Source: orchestration system tables (job-run timelines, task-level events) + native
    retries/repair. Metrics: run success rate, duration p95, queue wait, task retries,
    repair-run frequency per pipeline. Rooted in the orchestrator - that is where retries
    and dependencies already live.""",
  "L2 DATA HEALTH (is the output any good?)": """
    Freshness: last commit ts vs SLA (from DESCRIBE HISTORY / table metrics).
    Volume: row count + bytes vs a trailing 30-day band (robust z / MAD, not raw min-max).
    Schema: schema hash diff vs the data contract (additive vs breaking).
    Quality: nulls, dupes, referential integrity from monitors/expectations.
    Everything is EMITTED AS ROWS into per-table metrics Delta tables: queryable, cheap,
    replayable - and exactly what alerts evaluate.""",
  "L3 COST (can we afford this?)": """
    DBU/spend per pipeline from billing system tables joined to custom tags (team, product).
    Chargeback dashboards + cost-per-refresh trend; alert on UNIT-COST anomalies
    (cost per million rows), not just absolute spend.""",
  "ALERT ROUTING": """
    S1 = customer SLA breached/impending -> PagerDuty + incident channel.
    S2 = internal SLA missed -> team channel, business hours.
    S3 = trend anomaly -> weekly digest.
    Every alert MUST carry an owning team tag and a runbook link - an alert without an
    owner is noise by definition.""",
  "SYMPTOM vs CAUSE": """
    Alert on SYMPTOMS (table is 2h stale / SLA will breach in 30 min) - a pipeline can fail
    50 ways but customers feel only a few symptoms. Causes (task failed, backpressure,
    upstream late) are diagnostic context attached to the symptom page - not separate pages.
    One incident -> one page, no matter how many tasks failed underneath.""",
  "ANTI-NOISE": """
    - Alert BUDGET: each team gets N pages/week; exceeded -> prune the lowest-value alerts.
    - Flapping control: page only on sustained breach (2 consecutive evaluations).
    - Dedup by (table, pipeline) - never by (task, timestamp).
    - Weekly review: top noisiest alerts get fixed, tuned, or deleted.
    Target: <5 actionable pages/week platform-wide. More than that and on-call ignores
    real ones - which is the actual failure mode of observability stacks.""",
  "ROLLOUT": """
    Weeks 1-2: metrics tables + freshness for the top-20 revenue tables. Weeks 3-4:
    volume/schema bands + severity routing. Month 2: quality expectations + cost dashboards.
    Start from the symptoms customers already complain about - not a 3,000-table big bang.""",
}
for k, v in BLUEPRINT.items():
    print(f"### {k}\n{v}\n")

example_queries = """
-- freshness (the SYMPTOM customers feel), from the metrics table the monitors fill
SELECT table_name,
       MAX(committed_at) AS last_commit,
       DATEDIFF(minute, MAX(committed_at), CURRENT_TIMESTAMP()) AS staleness_min,
       MAX(sla_minutes) AS sla_min
FROM ops.table_freshness_metrics
GROUP BY table_name
HAVING DATEDIFF(minute, MAX(committed_at), CURRENT_TIMESTAMP()) > MAX(sla_minutes);

-- volume band breach (MAD-based: robust to outliers, only real incidents page)
SELECT table_name, dt, rows, robust_z
FROM ops.table_volume_metrics
WHERE ABS(robust_z) > 6;
"""
print(example_queries)
print("Q18 OK — three metric layers + symptom-first routing + a noise budget")

## Day 19 — Score Card

Score each question **0-3** after attempting it out loud (before reading the solution):

* **0** — couldn't answer · **1** — needed major hints · **2** — solid with minor gaps · **3** — interview-ready, named the trade-offs unprompted

| Q | Category | Topic | Level | Score (0-3) |
| --- | --- | --- | --- | --- |
| Q1 | SQL Advanced | Time-series gaps — spine + LOCF | ADVANCED | |
| Q2 | SQL Advanced | Fuzzy entity resolution (levenshtein + blocking) | ADVANCED | |
| Q3 | Python | Monotonic stack — next warmer reading | ADVANCED | |
| Q4 | Python | Union-Find — merge duplicate CRM accounts | HARD | |
| Q5 | PySpark | Arbitrary stateful sessionization | ADVANCED | |
| Q6 | PySpark | Stream-to-stream join, dual watermarks | ADVANCED | |
| Q7 | Spark Optimization | Executor memory anatomy + GC tuning | ADVANCED | |
| Q8 | Spark Optimization | Sort-merge vs shuffled hash join | HARD | |
| Q9 | ETL | Blue-green expand-contract schema migration | ADVANCED | |
| Q10 | ETL | DEEP CLONE for DR & env seeding | ADVANCED | |
| Q11 | ETL | Checkpoint-safe streaming upgrades | HARD | |
| Q12 | Data Modeling | One Big Table vs star schema | ADVANCED | |
| Q13 | Data Modeling | Ragged hierarchy flattening | ADVANCED | |
| Q14 | Snowflake | Data sharing architecture | ADVANCED | |
| Q15 | Snowflake | Event tables & Snowflake Logging | ADVANCED | |
| Q16 | Monitoring | For-Each fan-out + task values | ADVANCED | |
| Q17 | Monitoring | pytest + chispa pipeline testing | ADVANCED | |
| Q18 | Monitoring | Observability stack design | HARD | |

**Review rules**:

* Any question scoring **< 2** → re-attempt it tomorrow morning BEFORE starting Day 20.
* Two questions **< 2** in the same category → reread that category's Day 15-18 questions too; the gap is foundational.
* Keep every answer under **4 minutes** — length signals lack of judgment, not depth.
* Log today's total: ____ / 54. The trend line across the week matters more than any single day.

> **Tomorrow — Day 20 (Week 3):** deeper system-design judgment — multi-tenant isolation, cost governance at platform scale, and more 2025-2026 trending topics (streaming Lakehouse, GenAI data pipelines). Days 22+ go full FAANG level.